In [ ]:
# ==== Config ====
SCRIPT_TEXT = """Hello, and welcome to this channel.
Today, I will read this script aloud, and my lips will move in sync with every word.
Let's get started."""
SCRIPT_PATH = None      # e.g. "/kaggle/input/my-script/script.txt" (overrides SCRIPT_TEXT)
AVATAR_IMAGE = "/kaggle/input/vault-live-input/apresentador.png"     # e.g. "/kaggle/input/my-avatar/face.png"; None -> sample image
SPEAKER_WAV = None      # e.g. "/kaggle/input/my-voice/ref.wav" (voice clone); None -> built-in XTTS speaker
XTTS_SPEAKER = "Ana Florence"  # built-in voice used when SPEAKER_WAV is None
LANGUAGE = "en"         # XTTS language code: en, ja, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, hu, ko, hi

PAUSE_SEC = 0.35        # silence inserted between sentences
SIZE = 512              # SadTalker face render size: 256 (faster) or 512 (sharper)
PREPROCESS = "full"     # "full" keeps the whole image, "crop" outputs the face only
STILL = True            # True = less head motion (recommended with "full")
ENHANCER = "gfpgan"     # "gfpgan" or None (faster)
POSE_STYLE = 0          # 0-45, head-motion style
EXPRESSION_SCALE = 1.0  # >1 = stronger mouth/expression
BATCH_SIZE = 2          # face render batch size; halved automatically on CUDA OOM (512px on T4: 2)

WORK = "/kaggle/working"
SADTALKER_DIR = f"{WORK}/SadTalker"
import glob
NARRATION = (glob.glob("/kaggle/input/**/fala.wav", recursive=True) or [None])[0]
AVATAR_IMAGE = (glob.glob("/kaggle/input/**/apresentador.png", recursive=True) or [AVATAR_IMAGE])[0]
print("inputs:", NARRATION, AVATAR_IMAGE)
ENHANCER = None
SIZE = 256


In [ ]:
# ==== Install ====
import subprocess, sys, os

def sh(cmd):
    # Run a shell command and fail loudly
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

# Pin the preinstalled torch stack so pip does not replace it
sh(f"{sys.executable} -m pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > {WORK}/constraints.txt || true")



sh(f"{sys.executable} -m pip install -q -c {WORK}/constraints.txt yacs kornia pydub safetensors imageio imageio-ffmpeg scikit-image basicsr facexlib gfpgan")
if LANGUAGE == "ja":
    # Japanese text processing for XTTS
    sh(f"{sys.executable} -m pip install -q cutlet fugashi unidic-lite")

In [ ]:
# ==== SadTalker: clone, download weights, patch for new numpy/torchvision ====
import os, re, glob, site

if not os.path.isdir(SADTALKER_DIR):
    sh(f"git clone -q --depth 1 https://github.com/OpenTalker/SadTalker.git {SADTALKER_DIR}")

CKPT = {
    "checkpoints/mapping_00109-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar",
    "checkpoints/mapping_00229-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar",
    "checkpoints/SadTalker_V0.0.2_256.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors",
    "checkpoints/SadTalker_V0.0.2_512.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors",
    "gfpgan/weights/alignment_WFLW_4HG.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
    "gfpgan/weights/detection_Resnet50_Final.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    "gfpgan/weights/GFPGANv1.4.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    "gfpgan/weights/parsing_parsenet.pth": "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
}
for rel, url in CKPT.items():
    dst = os.path.join(SADTALKER_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(dst) == 0:
        sh(f"wget -q -O {dst} {url}")

def patch(path, old, new):
    # Replace a code snippet once; skip if already patched
    src = open(path).read()
    if new in src:
        return
    assert old in src, f"patch target not found in {path}: {old}"
    open(path, "w").write(src.replace(old, new))
    print("patched", path)

# numpy >= 1.24: np.float removed, ragged np.array raises
patch(f"{SADTALKER_DIR}/src/face3d/util/my_awing_arch.py",
      "preds.astype(np.float, copy=False)", "preds.astype(np.float64, copy=False)")
patch(f"{SADTALKER_DIR}/src/face3d/util/preprocess.py",
      "t = np.stack([sTx, sTy], axis=0)\n",
      "t = np.stack([sTx, sTy], axis=0).reshape(-1)  # flatten (2,1)->(2,) for numpy>=1.25\n")
patch(f"{SADTALKER_DIR}/src/utils/preprocess.py",
      "np.array([float(item) for item in np.hsplit(trans_params, 5)])",
      "np.asarray(trans_params, dtype=np.float64).reshape(-1)[:5]")
# numpy 2.x: VisibleDeprecationWarning moved to np.exceptions
patch(f"{SADTALKER_DIR}/src/face3d/util/preprocess.py",
      "category=np.VisibleDeprecationWarning",
      "category=getattr(np, 'VisibleDeprecationWarning', DeprecationWarning)")

# torchvision >= 0.17 removed functional_tensor (used by basicsr)
for sp in site.getsitepackages():
    for f in glob.glob(f"{sp}/basicsr/data/degradations.py"):
        patch(f, "torchvision.transforms.functional_tensor import", "torchvision.transforms.functional import")
# Keep rendered frames on CPU so GPU memory does not grow with video length
patch(f"{SADTALKER_DIR}/src/facerender/modules/make_animation.py",
      "predictions.append(out['prediction'])\n",
      "predictions.append(out['prediction'].cpu())\n")
print("SadTalker ready")

In [ ]:
# ==== Step 2: narration.wav + avatar -> lip-synced video with SadTalker ====
import glob, os, shutil, subprocess, sys, torch

avatar = AVATAR_IMAGE or f"{SADTALKER_DIR}/examples/source_image/art_3.png"
result_dir = f"{WORK}/sadtalker_results"
cmd = [sys.executable, "inference.py",
       "--driven_audio", NARRATION,
       "--source_image", avatar,
       "--result_dir", result_dir,
       "--size", str(SIZE),
       "--preprocess", PREPROCESS,
       "--pose_style", str(POSE_STYLE),
       "--expression_scale", str(EXPRESSION_SCALE),
       ]
if STILL:
    cmd.append("--still")
if ENHANCER:
    cmd += ["--enhancer", ENHANCER]
if not torch.cuda.is_available():
    cmd.append("--cpu")
env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True")
bs = BATCH_SIZE
while True:
    # Retry with a smaller batch if rendering fails (typically CUDA OOM)
    run = cmd + ["--batch_size", str(bs)]
    print(" ".join(run))
    if subprocess.run(run, cwd=SADTALKER_DIR, env=env).returncode == 0:
        break
    if bs == 1:
        raise RuntimeError("SadTalker failed even with batch_size=1 (try SIZE=256 or ENHANCER=None)")
    bs = max(1, bs // 2)
    print(f"retrying with batch_size={bs}")

latest = max(glob.glob(f"{result_dir}/*.mp4"), key=os.path.getmtime)
OUTPUT = f"{WORK}/talking_avatar.mp4"
shutil.copy(latest, OUTPUT)
print("output:", OUTPUT)